# 02 — Per-signal evaluation & decay

Statistical test (IC/ICIR, Fama–MacBeth) → economic test (quintile long-short, gross and net) → decay segmentation (McLean–Pontiff). Evaluation **precedes** modeling; the placebo `sig_dead` must fail.

In [ ]:
# Path shim: make the repo root importable when running from notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.width", 140)


In [ ]:
import yaml
from src.data.synthetic import make_synthetic_panel
from src.utils.stats import rank_normalize_cross_section

cfg = yaml.safe_load(open(ROOT / "configs/config.yaml"))
panel, factors, meta = make_synthetic_panel(cfg, seed=cfg["run"]["seed"])
signal_cols = list(meta.index)
panel = rank_normalize_cross_section(panel, signal_cols)
print(panel["date"].nunique(), "months x", panel["ticker"].nunique(), "names")
meta


## IC, ICIR, and the quintile long-short

In [ ]:
from src.evaluation.ic import mean_ic
from src.evaluation.portfolio import evaluate_signal_portfolio, summary_row
import pandas as pd

rows, series = [], {}
for c in signal_cols:
    res = evaluate_signal_portfolio(panel, c,
        n_q=cfg['evaluation']['n_quantiles'],
        cost_bps_per_side=cfg['evaluation']['cost_bps_per_side'])
    icr = mean_ic(panel, c)
    row = summary_row(res); row.update(IC=icr['ic_mean'],
        IC_t=icr['ic_tstat'], ICIR=icr['icir'])
    rows.append(row); series[c] = res['series']['gross']
tab = pd.DataFrame(rows).set_index('signal')
tab.round(3)

Read the table against the planted truth in `meta`: net Sharpe ordering should follow `true_beta`, and `sig_dead` should be indistinguishable from zero (|t| < 2).

## Fama–MacBeth: marginal predictive power

In [ ]:
from src.evaluation.fama_macbeth import fama_macbeth
fama_macbeth(panel, signal_cols).round(4)

## Decay: in-sample vs post-sample vs post-publication

In [ ]:
from src.evaluation.decay import decay_table
decay_table(series, meta).round(3)

In [ ]:
import matplotlib.pyplot as plt
lead = tab['sharpe_gross'].idxmax()
ls = series[lead]
ax = ls.cumsum().plot(figsize=(9, 4), lw=1.4,
    title=f'{lead}: cumulative L/S with decay markers')
ax.axvline(meta.loc[lead, 'sample_end'], ls='--', color='orange', label='sample end')
ax.axvline(meta.loc[lead, 'pub_date'], ls='--', color='red', label='publication')
ax.legend(); plt.show()